# SceneTIQ backend on Colab

Starts the SceneTIQ API on this Colab GPU, stores every Scene on Google Drive, and exposes it at a stable URL via a Cloudflare **named tunnel** (ADR 0004).

**Before the first run** (see `AGENTS.md` → *Colab*):
1. Add Colab secrets (🔑 sidebar): `SCENETIQ_TOKEN` (any long random string) and `CLOUDFLARE_TUNNEL_TOKEN` (from your named tunnel).
2. In the Cloudflare dashboard, route the tunnel's public hostname to `http://localhost:8000`.
3. Runtime → Change runtime type → GPU (A100 80GB).

Run all cells. Re-running after a runtime restart brings back the same Scenes at the same URL; jobs that were running are marked failed.

In [ ]:
# Parameters
REPO_URL = "https://github.com/luatducvo/scenetiq.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/scenetiq"   # Scenes + SQLite live here
PIPELINE = "fake"                                # "fake" (no models) or "real" (GPU model stack)
PORT = 8000

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import subprocess, pathlib
repo = pathlib.Path("/content/scenetiq")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
extra = "[real]" if PIPELINE == "real" else ""
subprocess.run(["pip", "install", "-q", "-e", f"{repo}{extra}"], check=True)

In [ ]:
from pathlib import Path
from google.colab import userdata
from scenetiq import colab

token = userdata.get("SCENETIQ_TOKEN")
tunnel_token = userdata.get("CLOUDFLARE_TUNNEL_TOKEN")
storage_root = Path(DRIVE_ROOT) / "data"
logs = Path(DRIVE_ROOT) / "logs"

env = colab.server_environment(token, storage_root, PIPELINE)
server = colab.start_server(env, PORT, logs / "server.log")
colab.wait_until_ready(PORT, token, timeout=600)
print("backend ready on port", PORT)

In [ ]:
tunnel = colab.start_named_tunnel(tunnel_token, logs / "tunnel.log")
print("tunnel started; open the frontend and set the backend URL to your tunnel hostname")

In [ ]:
# Tail logs (re-run to refresh)
print((logs / "server.log").read_text()[-4000:])